# 09 - Create Data Splits (Stratified by Signer)

This notebook implements a robust data splitting strategy (Strategy 1) to create training, validation, and test sets.

**Strategy: Stratified Group Split by Signer ID**
- **Goal**: To ensure that the model is evaluated on its ability to generalize to unseen signers.
- **Method**:
  1. The dataset is split based on `signer_id`. All videos from a particular signer will belong to *only one* set (train, val, or test).
  2. The split is **stratified** by `label`. This ensures that the class distribution is preserved as much as possible across the train, validation, and test sets.
- **Process**:
  1. Load the cleaned, top-N class dataset (before augmentation).
  2. Use `StratifiedGroupKFold` from `scikit-learn` to perform a two-step split:
     - First, an 80/20 split to separate the training set from a temporary hold-out set.
     - Second, a 50/50 split on the hold-out set to create the final validation and test sets.
  3. Add a new `split` column to the dataframe.
  4. Save the final dataframe with split information for use in model training.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

: 

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

# Use the output from notebook 06 (Top-N classes, before normalization and augmentation)
INPUT_CSV = PROJECT_ROOT / 'merged_datasets' / 'universal_metadata_topn_classes.csv'
OUTPUT_CSV = PROJECT_ROOT / 'merged_datasets' / 'universal_metadata_topn_with_splits.csv'

# Configure the splits
N_SPLITS_TRAIN_TEST = 5  # Results in an 80/20 split
N_SPLITS_VAL_TEST = 2    # Splits the 20% group into two 10% groups
RANDOM_SEED = 42

print('INPUT_CSV:', INPUT_CSV)
print('OUTPUT_CSV:', OUTPUT_CSV)

## Load and Prepare Data

Load the dataset and filter out any rows where `signer_id` is missing, as they cannot be used for a grouped split.

In [ ]:
df = pd.read_csv(INPUT_CSV)

print('Original shape:', df.shape)
print('Original unique labels:', df['label'].nunique())
print('Original unique signers:', df['signer_id'].nunique())

# Drop rows where signer_id is NaN, as we cannot group them.
df_clean = df.dropna(subset=['signer_id']).copy()
df_clean['signer_id'] = df_clean['signer_id'].astype(int)

print('\\nShape after dropping NaN signers:', df_clean.shape)
print('Signers remaining:', df_clean['signer_id'].nunique())

X = df_clean
y = df_clean['label']
groups = df_clean['signer_id']

## Perform the Stratified Group Split

We'll use `StratifiedGroupKFold` in two stages to achieve an 80/10/10 split.

In [ ]:
# Step 1: Split into training (80%) and a temporary test/val set (20%)
sgkf_train_test = StratifiedGroupKFold(n_splits=N_SPLITS_TRAIN_TEST, shuffle=True, random_state=RANDOM_SEED)
train_idx, temp_idx = next(sgkf_train_test.split(X, y, groups))

# Create the initial split in the dataframe
df_clean['split'] = ''
df_clean.iloc[train_idx, df_clean.columns.get_loc('split')] = 'train'
df_clean.iloc[temp_idx, df_clean.columns.get_loc('split')] = 'temp'

# Prepare the temporary dataframe for the second split
temp_df = df_clean[df_clean['split'] == 'temp']
X_temp = temp_df
y_temp = temp_df['label']
groups_temp = temp_df['signer_id']

# Step 2: Split the temporary set (20%) into validation (10%) and test (10%)
sgkf_val_test = StratifiedGroupKFold(n_splits=N_SPLITS_VAL_TEST, shuffle=True, random_state=RANDOM_SEED)
# We need to get the indices relative to the original dataframe to assign them correctly
val_relative_idx, test_relative_idx = next(sgkf_val_test.split(X_temp, y_temp, groups_temp))

# Get absolute indices from the temporary dataframe's index
val_abs_idx = temp_df.index[val_relative_idx]
test_abs_idx = temp_df.index[test_relative_idx]

# Assign the final splits
df_clean.loc[val_abs_idx, 'split'] = 'val'
df_clean.loc[test_abs_idx, 'split'] = 'test'

print("Split creation complete.")

## Verify the Splits

Let's check the statistics to ensure the split was successful. We need to confirm:
1. The number of samples in each set.
2. The number of unique signers in each set.
3. That there is **no overlap** of signers between the sets.

In [ ]:
split_stats = df_clean.groupby('split').agg(
    n_samples=('label', 'count'),
    n_signers=('signer_id', 'nunique')
)

print("--- Split Statistics ---")
display(split_stats)

train_signers = set(df_clean[df_clean['split'] == 'train']['signer_id'].unique())
val_signers = set(df_clean[df_clean['split'] == 'val']['signer_id'].unique())
test_signers = set(df_clean[df_clean['split'] == 'test']['signer_id'].unique())

print("\\n--- Signer Overlap Check ---")
print(f"Train ∩ Val: {len(train_signers.intersection(val_signers))}")
print(f"Train ∩ Test: {len(train_signers.intersection(test_signers))}")
print(f"Val ∩ Test: {len(val_signers.intersection(test_signers))}")

if len(train_signers.intersection(val_signers)) == 0 and len(train_signers.intersection(test_signers)) == 0 and len(val_signers.intersection(test_signers)) == 0:
    print("\\n✅ Verification successful: No signer overlap between sets.")
else:
    print("\\n❌ Verification failed: Signer overlap detected!")

# Check class distribution
print("\\n--- Class Distribution per Split (Top 10 Classes) ---")
class_dist = df_clean.groupby('split')['label'].value_counts(normalize=True).mul(100).rename('percent').reset_index()
display(class_dist.groupby('split').head(10))

## Save the Final Dataset

Save the dataframe with the new `split` column to a CSV file.

In [ ]:
# Sort for consistency
df_final = df_clean.sort_values(['label', 'signer_id']).reset_index(drop=True)

df_final.to_csv(OUTPUT_CSV, index=False)

print(f"Successfully saved dataset with splits to: {OUTPUT_CSV}")
display(df_final.head())